# Inspect Docling Extraction

View the raw extracted JSON for any page of any indexed document.

In [ ]:
import json
import sqlite3
from pathlib import Path

DATA_DIR = Path("../boardgame_agent/data")
DB_PATH = DATA_DIR / "games.db"

In [ ]:
# ── Set these ─────────────────────────────────────────────
GAME_ID = "the_crew__the_quest_for_planet_nine"
DOC_TAG = None  # filter by tag, or None to show all docs
PAGE_NUM = 4
# ──────────────────────────────────────────────────────────

In [ ]:
# List available games
games_dir = DATA_DIR / "games"
print("Available games:")
for g in sorted(games_dir.iterdir()):
    if g.is_dir():
        print(f"  {g.name}")

In [ ]:
# List all documents with their tags
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row
rows = conn.execute(
    "SELECT game_id, doc_name, doc_tag FROM documents ORDER BY game_id, doc_tag, doc_name",
).fetchall()
conn.close()

print("All documents and tags:")
current_game = None
for r in rows:
    if r["game_id"] != current_game:
        current_game = r["game_id"]
        print(f"\n  {current_game}")
    print(f"    {r['doc_name']} ({r['doc_tag']})")

In [ ]:
# List documents for the selected game
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row
rows = conn.execute(
    "SELECT doc_name, doc_tag FROM documents WHERE game_id = ? ORDER BY doc_name",
    (GAME_ID,),
).fetchall()
conn.close()

doc_tags = {r["doc_name"]: r["doc_tag"] for r in rows}
print(f"Documents for {GAME_ID}:")
for name, tag in doc_tags.items():
    print(f"  {name} ({tag})")

In [ ]:
# Load extractions, filter by tag if set
extracted_dir = games_dir / GAME_ID / "extracted"
target_docs = (
    {name for name, tag in doc_tags.items() if tag == DOC_TAG}
    if DOC_TAG else set(doc_tags.keys())
)

all_pages = []
for f in sorted(extracted_dir.glob("*.json")):
    if f.stem in target_docs:
        pages = json.loads(f.read_text())
        all_pages.extend(pages)
        print(f"Loaded: {f.stem} ({len(pages)} pages)")

# Find the requested page
page = next((p for p in all_pages if p["page_num"] == PAGE_NUM), None)
if page is None:
    available = sorted(set(p["page_num"] for p in all_pages))
    print(f"\nPage {PAGE_NUM} not found. Available: {available}")
else:
    print(f"\n{'='*60}")
    print(f"doc_name: {page['doc_name']}")
    print(f"page_num: {page['page_num']}")
    print(f"bboxes:   {len(page.get('bboxes', []))}")
    print(f"{'='*60}\n")
    print(json.dumps(page, indent=2))